# Cleanup and Resources

In this lesson, you will learn to arrange cleanup when Java operations finish, return a value, or fail, and to keep useful information when cleanup also fails.

CSC-239 · Module 7 · Lesson 3 of 4

An exception handler can explain a failed operation, but that response does not automatically release everything the operation was using. You will make resource lifetimes visible with small classroom models, then trace cleanup through successful and failed paths.

Use the [Module 7 glossary](terms.md) to revisit the vocabulary after reading its explanation.

## Learning Goals

- Trace `finally` during normal completion, a pending return, and an exception.
- Implement an observable `AutoCloseable` model and use try-with-resources to close resources in reverse initialization order.
- Distinguish a primary failure from suppressed cleanup failures and explain how to inspect both.

## Why This Matters

Applications often use limited resources while doing a job. An open file stream, for example, uses resources beyond an ordinary Java variable. Once the application is finished with it, closing it releases those resources. If failed operations repeatedly leave resources open, later requests may eventually be unable to get what they need.

Putting a cleanup call at the bottom of an ordinary block is insufficient: a return or exception can skip that line. Cleanup needs to be connected to leaving the operation, including its failed paths. This is separate from deciding what message to show a user.

You already know how exceptions travel to handlers. This lesson adds cleanup to that path. The next module will apply the pattern to real files; here, printed lifecycle messages let you focus on the order before file operations add more details.

## Check Your Starting Point

Read the complete snippet without running it. Predict every output line, and explain which path the supplied count selects. Decide whether the method returns a value and whether the caller prints its `Accepted` message. Identify the first statement reached after the handler. This recalls method failure and caller control flow before we add resource cleanup.

```java
class StartingCheck {
    public static int requirePositive(int count) {
        if (count < 1) {
            throw new IllegalArgumentException("Count must be positive.");
        }
        return count;
    }
}
try {
    System.out.println("Count: " + StartingCheck.requirePositive(0));
    System.out.println("Accepted");
} catch (IllegalArgumentException problem) {
    System.out.println(problem.getMessage());
}
System.out.println("After request");
```

In [ ]:
Predicted output in order:

Path selected by the supplied count:

Whether a value returns and Accepted prints, with reasons:

First statement reached after the handler:


<details><summary>Show answer: follow the method contract</summary>

```text
Count must be positive.
After request
```

The zero count takes the throw branch and leaves the method before return. The failed method call interrupts the Count print, and Accepted is skipped. The handler prints the message; execution then prints After request. This retrieves failure control flow before adding cleanup.

</details>

## Video Demonstration

Trace the resource declarations from left to right and the close operations from right to left. Predict the whole message sequence before the command runs.

<video controls preload="metadata" width="960">
  <source src="media/03_cleanup_and_resources/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_cleanup_and_resources/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the cleanup and resources demonstration transcript](media/03_cleanup_and_resources/transcript.md).


## Concept

### Decide where use begins and ends

A **resource lifetime** is the interval from acquiring a resource to releasing it. Imagine campus workshop staff using a map and a badge while preparing a request. They need to finish using both even if the request cannot be completed.

Our Java objects model those stages with messages. Constructing a `NamedResource` prints that its modeled use began; calling `close` prints that its modeled use ended. These objects do not open real files or borrow physical equipment. That limit matters: a real resource's close operation must do the actual release work, not just print a message.

We will first examine Java's general cleanup block. Then we will connect cleanup to an object through an interface, so Java can call the object's close operation automatically.

### Put cleanup on the exit path

The Java keyword **`finally`** introduces a block that normally executes as control leaves the associated `try` and any matching `catch`. The block can run after ordinary completion, during a return, or while an exception is leaving.

```java
try {
    System.out.println("Begin");
    throw new IllegalStateException("Work stopped.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
} finally {
    System.out.println("Cleanup");
}
System.out.println("After");
```

The body prints `Begin` and raises an exception. **`IllegalStateException`** is an unchecked exception class used when an operation cannot proceed in its present state. The matching handler prints `Handled: Work stopped.` After that handler finishes, `finally` prints `Cleanup`. Only then does the following statement print `After`.

The handler and cleanup have different jobs. The handler responds to the failure; the cleanup block performs work needed when leaving the operation. Merely having a `catch` block would not cause a separate cleanup statement to run.

### Finish cleanup before delivering a return value

A return also leaves a method, so it must pass through an enclosing `finally` block:

```java
class CompletionTools {
    public static int finish() {
        try {
            System.out.println("Compute");
            return 7;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
System.out.println("Result: " + CompletionTools.finish());
```

The output is:

```text
Compute
Cleanup
Result: 7
```

Java evaluates the return expression first. The value `7` is ready, but the caller has not received it yet. The cleanup block runs next. Because that block completes normally, the pending return continues, and the caller can finish its result message.

Avoid adding a `return` or a new `throw` to `finally` simply to end cleanup. That new exit can replace a pending result or exception and hide the original outcome. Also, `finally` is not a promise that cleanup can run after the Java process is forcibly terminated. Our examples trace ordinary Java control flow.

<details class="animation-panel" open>
<summary>Finally pending return — show or hide animation</summary>
<p><img src="media/03_cleanup_and_resources/finally_pending_return.gif" alt="The return value is evaluated before finally executes. The caller has not received the result yet. Normal cleanup completes and the pending return continues. Cleanup also runs as the exception leaves the protected body. The pending return or exception continues. A new abrupt exit from finally could replace it." width="960" style="max-width:100%;height:auto;"></p>
</details>

The result is ready before cleanup starts, but the caller receives it only after finally finishes normally. The separate failed path also passes through cleanup. The final comparison shows that normally completing cleanup preserves the pending outcome; a new return or throw in finally could replace it. The loop lasts about 13 seconds. Hide it with the control above or use the [static finally pending return diagram](media/03_cleanup_and_resources/finally_pending_return_still.png).

### Give an object an automatic cleanup operation

**`AutoCloseable`** is a Java interface with a `close` method. Implementing it supplies a standard operation Java can invoke when the resource's scope ends. It is a type name, not a keyword, and it belongs to `java.lang`, so no import is needed.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
```

The private field keeps the name for this object. The constructor stores its argument, then prints the start message. When `close()` runs later, it uses that same field to identify which modeled resource is closing.

The `implements AutoCloseable` declaration connects the class to the interface. The public `close` method supplies its cleanup behavior, and `@Override` asks the compiler to check the inherited method relationship. It does not call the method. Defining this class alone does not create a resource or schedule cleanup.

The interface permits `close` to declare a checked `Exception`. This particular implementation only prints a message, so its own declaration needs no `throws` clause. An implementation can meet an interface contract without declaring every failure the interface permits.

### Register the object for cleanup

A **try-with-resources statement** arranges automatic cleanup for successfully initialized `AutoCloseable` resources. Declare the resource inside parentheses immediately after `try`:

```java
try (NamedResource resource = new NamedResource("notes")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Java first constructs the object and stores its reference in `resource`. That construction prints `Opened: notes`. The body then prints `Work`. When the body finishes, Java calls `resource.close()`, which prints `Closed: notes`. The following statement finally prints `Done`.

The declaration in the resource header is what connects this object to automatic cleanup. The variable declared there is available in the `try` body; it is not a variable for the following statements to keep using. You do not add a manual `close()` call at the end of the body.

Automatic cleanup also takes place if the body throws an exception. Java attempts the close operation before an attached handler responds. That prevents a failed body from skipping cleanup just because it never reaches its final ordinary statements.

<details class="animation-panel" open>
<summary>Resource lifetime — show or hide animation</summary>
<p><img src="media/03_cleanup_and_resources/resource_lifetime.gif" alt="Constructor prints Opened: notes. This model prints lifecycle events; it does not open a real file. The resource is in use within the try-with-resources body. Java invokes close as the protected scope ends. The close method runs before the following statement." width="960" style="max-width:100%;height:auto;"></p>
</details>

The header creates the notes resource before Work. Leaving the body invokes close automatically, so Closed: notes appears before Done. The same object is used throughout; these messages model a lifetime without opening a real file. The loop lasts about 10.53 seconds. Hide it with the control above or use the [static resource lifetime diagram](media/03_cleanup_and_resources/resource_lifetime_still.png).

### Close later resources before earlier ones

A single try-with-resources statement can declare more than one resource. Separate the declarations with a semicolon:

```java
try (NamedResource first = new NamedResource("first");
     NamedResource second = new NamedResource("second")) {
    System.out.println("Work");
}
```

Initialization goes from left to right: `first`, then `second`. **Reverse resource closure** means cleanup goes in the opposite order: `second`, then `first`. Names do not determine that order; the positions of the successfully initialized resources do.

Closing later resources first is useful when they depend on earlier ones. The earlier resource stays available while the later one finishes using it. If the second resource's initialization fails, the first resource still needs cleanup. Java closes the resources that were successfully initialized before that failure; it does not run the body or call `close` on an object whose resource initialization did not complete.

For the successful two-resource example, trace construction, construction, work, second close, first close. If the body fails, those close attempts still happen before an attached handler responds.

<details class="animation-panel" open>
<summary>Reverse close order — show or hide animation</summary>
<p><img src="media/03_cleanup_and_resources/reverse_close_order.gif" alt="first is now eligible for automatic cleanup. Both resources are available to the body. The body finishes and cleanup begins. The later initialized resource closes first. Both close calls have completed normally. The body is never entered. first closes; no close call is made for the resource whose initialization failed." width="960" style="max-width:100%;height:auto;"></p>
</details>

Initialization moves from first to second, while closing moves from second to first. The separate constructor-failure case shows why only successfully initialized resources receive automatic close calls. When second fails to initialize, first still closes and the body never begins. The loop lasts about 15.53 seconds. Hide it with the control above or use the [static reverse close order diagram](media/03_cleanup_and_resources/reverse_close_order_still.png).

### Preserve a work failure when cleanup also fails

Cleanup can encounter its own problem. Suppose the body already failed, and a close operation then raises another exception. The **primary exception** is the failure that continues outward to the caller. A **suppressed exception** is an additional failure attached to it during this cleanup process.

In try-with-resources, a body exception remains primary when automatic closing also fails. This retains the original reason the work stopped while preserving the cleanup diagnostic for inspection. Suppressed does not mean that the extra failure has vanished.

```java
class FailingResource implements AutoCloseable {
    public FailingResource() {
        System.out.println("Opened");
    }
    @Override
    public void close() {
        System.out.println("Closing");
        throw new IllegalStateException("Close failed.");
    }
}
try (FailingResource resource = new FailingResource()) {
    throw new IllegalArgumentException("Body failed.");
} catch (IllegalArgumentException problem) {
    System.out.println("Primary: " + problem.getMessage());
    for (Throwable secondary : problem.getSuppressed()) {
        System.out.println("Suppressed: " + secondary.getMessage());
    }
}
```

Construction prints `Opened`. The body raises its `IllegalArgumentException`. Before that object reaches the handler, Java calls `close`, which prints `Closing` and raises an `IllegalStateException`. The body exception stays primary, so the specific `IllegalArgumentException` handler receives it.

The output is:

```text
Opened
Closing
Primary: Body failed.
Suppressed: Close failed.
```

`getSuppressed()` returns an array of attached exceptions. The enhanced `for` loop visits each array entry and reads its message through the `Throwable` reference. If no secondary failure was attached, the array is empty and the loop prints no `Suppressed` lines.

A cause and a suppressed exception describe different relationships. In the previous lesson, a cause retained an earlier parsing failure inside a new application exception. Here, the suppressed exception comes from additional cleanup attempted after the primary failure. If the body completes normally and only `close` fails, that close failure can itself become the primary exception; there is no body failure to preserve ahead of it.

<details class="animation-panel" open>
<summary>Primary and suppressed — show or hide animation</summary>
<p><img src="media/03_cleanup_and_resources/primary_and_suppressed.gif" alt="A is the pending failure when automatic cleanup starts. Java retains both failures. The body failure stays primary; the cleanup failure is attached as suppressed, not as its cause. With no body failure pending, B becomes the primary failure. Report the main failure while retaining cleanup diagnostics." width="960" style="max-width:100%;height:auto;"></p>
</details>

A begins in the body and B begins in close. When both fail, the handler receives A with B attached as suppressed information. In the separate successful-body case, there is no A, so B becomes primary. The two inspection methods reveal the primary message and its attached cleanup failures. The loop lasts about 13 seconds. Hide it with the control above or use the [static primary and suppressed diagram](media/03_cleanup_and_resources/primary_and_suppressed_still.png).

## Worked Example

### Step 1: make each lifetime observable

The complete program below defines `NamedResource` with a stored name, a constructor message, and a close message. Each new object keeps its own name, so the output identifies the object involved in each stage. This is the same printed classroom model introduced above.

### Step 2: acquire both resources in one protected scope

The resource header constructs `first` and then `second`. Both declarations are inside the parentheses after `try`, so both successfully initialized objects are registered for automatic cleanup. The body prints `Work` only after both constructors finish.

### Step 3: compare work order with cleanup order

When the body ends, Java calls `close` on `second` and then `first`. The program places `Done` after the entire statement, so it cannot print until both normal close calls finish. Run the complete example and match each message to construction, body work, cleanup, or continuation.

In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("first");
     NamedResource second = new NamedResource("second")) {
    System.out.println("Work");
}
System.out.println("Done");


The complete output is:

```text
Opened: first
Opened: second
Work
Closed: second
Closed: first
Done
```

The first two lines show left-to-right initialization. `Work` appears after both objects are ready. The two close lines reverse the initialization order, and `Done` marks continuation after cleanup.

There are no explicit calls to `first.close()` or `second.close()` in the body. The resource header arranges those calls. Adding manual calls would duplicate the model's close messages and can be incorrect for real resources whose close behavior is not safe to repeat.

Next, the practice uses named resources and controlled failures to make the same order visible on several paths. Keep separate questions in mind: what ended the work, what cleanup Java attempted, and which failure reached the handler.

## Guided Practice

### Predict the modeled resource lifetime

The next complete program models two resources named `map` and `badge`. Its constructor and `close` method print messages so you can follow their lifetimes; it does not borrow physical items. Read the program before running it.

Predict every output line in order. Label each line as construction, work inside the body, cleanup, or the final caller statement. Explain what determines the order of the two `close` calls, and identify the beginning and end of each modeled lifetime. Keep this prediction when you run the cell.

In [ ]:
Predicted output, in order:

Role of each output line:

What determines close order:

Map lifetime begins and ends:
Badge lifetime begins and ends:


In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");


Run the entire `NamedResource` cell once. Record its complete output below without replacing your prediction. Identify the first difference, if any, and connect it to the statement or automatic cleanup call that produced that line. Which two lines show cleanup, and why does the final `Done` statement appear where it does?

In [ ]:
Actual output:

First difference from my prediction, or no difference:

Reason for that difference:

Cleanup calls in observed order:

Why Done appears at this point:


### Connect the interface to the cleanup statement

Use the program and your observed output to complete the table below. Number construction and closure separately, starting each sequence at 1. Then explain the different jobs of `implements AutoCloseable` and the resource declarations inside the `try` parentheses. Identify what makes a `close` operation available and what arranges for it to be called.

In [ ]:
Resource | Construction position | Close position | Final lifetime message
map      |                       |                |
badge    |                       |                |

The AutoCloseable contract supplies:

The try-with-resources statement arranges:


<details>
<summary>Show answer</summary>

The first constructor prints Opened: map, followed by Opened: badge from the second constructor. Both objects are successfully initialized in the try-with-resources header. Work prints in the body. When the body completes normally, Java calls close in reverse initialization order: badge, then map. Done appears after both calls finish. The name strings label the objects; they do not control the order. The AutoCloseable interface supplies the close contract, and these classes make the calls visible through print statements. They are classroom models and do not borrow actual items. The finally comparison and suppressed-failure model below extend this trace to other ways control leaves a block. A return or exception can skip later ordinary statements, so cleanup must be connected to leaving the protected work.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

Common error: Using the name strings to decide close order. Closing resources in the same order as construction. Placing Done before automatic cleanup.

</details>

### Predict cleanup during a return

The next complete program calls `CompletionTools.finish(false)`. The method uses a `finally` block around work that can return or fail. Read both the method and its caller before executing the cell.

Predict the full output in order. Explain when the return value is evaluated and when it can reach the caller. Use the placement of `finally` to justify your sequence.

In [ ]:
Predicted complete output for finish(false):

When the return value is evaluated:

What must complete before the caller receives it:


In [ ]:
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(false));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");


Run the complete `finish(false)` program and record its output. Compare it with your prediction. Explain the observed order of the cleanup message and the caller’s result message. Distinguish preparing a return value inside the method from completing that return to the caller.

In [ ]:
Actual output for finish(false):

First difference from my prediction, or no difference:

Why cleanup and the caller result appear in this order:


### Predict cleanup during a failure

The comparison program changes only the call to `finish(true)`. Read that complete program before running it. Follow the selected branch in the method, the `finally` block, and the caller’s matching handler.

Predict every output line. State whether the caller can print a `Result` line, and explain your decision using the method’s completion path. Do not run the cell until your prediction is recorded.

In [ ]:
Predicted complete output for finish(true):

Whether a Result line can print, and why:

Where cleanup belongs relative to the caller’s handler:


In [ ]:
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(true));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");


Run the complete `finish(true)` program. Record the output and compare it with your prediction. Explain how the failure leaves the method, when cleanup occurs, and why the caller follows its observed path.

These runs test ordinary return and exception paths while Java remains running. Explain why they do not establish a guarantee that cleanup runs after the Java process is abruptly terminated.

In [ ]:
Actual output for finish(true):

First difference from my prediction, or no difference:

Cleanup and caller-handler order, with a reason:

Why the failed call does or does not produce a Result line:

What these tests establish, and what they do not:


<details>
<summary>Show answer</summary>

With fail set to false, Compute prints and the method prepares to return 9. Before that return reaches the caller, finally prints Cleanup. The caller then prints Result: 9 and continues to After completion. With fail set to true, the body raises IllegalStateException before reaching the return. Finally still prints Cleanup as the failure leaves the method. The caller cannot complete its Result print; its specific catch prints Handled: Computation stopped. After completion follows. These are ordinary return and exception paths in a running Workspace; finally is not a guarantee against abrupt termination of the running Java program.

```java
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(false));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");
```

Expected output:

```text
Compute
Cleanup
Result: 9
After completion
```

Common error: Placing Result: 9 before Cleanup. Expecting the method to return 9 after it has thrown. Assuming finally only runs when the protected work fails.

**Check case 2.** The throw skips return 9, but leaving the try still runs finally. Cleanup prints before the exception reaches the caller's matching catch.

```java
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(true));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");
```

Expected output:

```text
Compute
Cleanup
Handled: Computation stopped.
After completion
```

</details>

### Predict a failure during automatic cleanup

The next `CleanupResource` program is a printed classroom model. Its `failOnClose` flag controls whether closing reports an additional failure. Read the complete class and caller before running it. The map flag is `false`, the badge flag is `true`, and the body explicitly throws after printing `Work`.

Predict every output line. Identify the exception raised by the body, the exception raised during closing, and the object you expect the handler to receive. Include both resources in your cleanup trace.

In [ ]:
Predicted complete output:

Body exception and its origin:

Cleanup exception and its origin:

Exception received by the handler:

Order of cleanup attempts:


In [ ]:
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");


Run the complete `CleanupResource` program once. Record its output and compare it with your prediction. Use the messages to distinguish the primary failure from any suppressed failure. Explain whether the map’s `close` call is attempted after the badge’s close operation fails, and connect that observation to automatic cleanup.

In [ ]:
Actual complete output:

First difference from my prediction, or no difference:

Primary message and where that failure began:

Suppressed message and where that failure began:

Evidence for which close calls were attempted, with a reason:


### Change only the cleanup flag

In the same complete program, change the badge constructor’s `failOnClose` argument from `true` to `false`. Keep the map flag, body exception, handler, and messages unchanged. Before rerunning, predict the full output and the length of the array returned by `getSuppressed()`. Explain what the enhanced `for` loop will print for that array.

In [ ]:
Predicted complete output after the one-flag change:

Predicted suppressed-array length:

What its traversal will print, and why:


Run the changed program and record its output. Explain what the suppressed-array length tells you about cleanup, and what it does not tell you about the body’s success. Contrast a suppressed cleanup failure with the earlier cause preserved in the previous lesson.

Then restore the badge flag to `true` and rerun the complete program. Record the restored result so the next comparison starts from the original case.

In [ ]:
Actual output with both close flags false:

Why the suppressed-message loop behaves this way:

What happened to the body failure:

A suppressed cleanup failure differs from an earlier cause because:

Actual output after restoring the badge flag to true:


<details>
<summary>Show answer</summary>

The resources open in the order map, badge. Work then raises the body exception with message Work failed. Automatic cleanup visits badge first; its close prints Closing: badge and raises another IllegalStateException. That later failure is attached to the original body exception as suppressed information. Java still calls close on map, which prints Closing: map and succeeds. The specific catch receives the primary body exception. getSuppressed returns an array with one attached cleanup failure, so the count is 1 and its message names badge. When both close flags are false, the body still fails and remains primary, but both cleanup calls succeed; the suppressed array is empty and the loop prints no Suppressed line. These are separate failure objects even though they have the same class. An earlier cause explains why a new exception was created; this suppressed exception records an additional failure during automatic cleanup.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closing: badge
Closing: map
Primary: Work failed.
Suppressed count: 1
Suppressed: Close failed: badge
After cleanup
```

Common error: Replacing the body message with the close failure message. Stopping the cleanup trace after badge close fails. Assuming an exception must have at least one suppressed failure. Using matching exception class names to conclude the two messages belong to the same object.

**Check case 2.** The body failure still reaches the catch after both resources close. No cleanup failure is attached, so getSuppressed returns an empty array and its traversal runs zero times.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", false)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closing: badge
Closing: map
Primary: Work failed.
Suppressed count: 0
After cleanup
```

</details>

### Complete the automatic cleanup structure

The supplied repair snippet is incomplete and is for reading only. Plan to copy the entire class and caller into the following Java work cell. Replace `RESOURCE_CONTRACT`, `CLOSE_OPERATION`, and `RESOURCE_SEPARATOR` using `AutoCloseable`, `close`, and `;`, each once. Keep the fields, constructors, declaration order, body, and printed messages unchanged.

Before writing and running the completed program, record your replacements, explain each one’s job, and predict the full output.

**Incomplete or faulty code for repair — read this example; write your corrected program in the Java work cell.**

```java
class NamedResource implements RESOURCE_CONTRACT {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void CLOSE_OPERATION() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map")RESOURCE_SEPARATOR
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```

In [ ]:
RESOURCE_CONTRACT replacement and job:
CLOSE_OPERATION replacement and job:
RESOURCE_SEPARATOR replacement and job:

Predicted complete output:


Run your complete repaired program. Record its output and the first difference from your prediction, if any. Identify the operation required by the interface and the exact part of the caller that registers both objects for automatic cleanup. If you made a correction, describe it and record the result of rerunning.

In [ ]:
Actual complete output:

Difference from my prediction, or no difference:

Operation required by the interface:

Where the resources are registered:

Correction and rerun result, if needed:


<details>
<summary>Show answer</summary>

The first constructor prints Opened: map, followed by Opened: badge from the second constructor. Both objects are successfully initialized in the try-with-resources header. Work prints in the body. When the body completes normally, Java calls close in reverse initialization order: badge, then map. Done appears after both calls finish. The name strings label the objects; they do not control the order. The AutoCloseable interface supplies the close contract, and these classes make the calls visible through print statements. They are classroom models and do not borrow actual items. RESOURCE_CONTRACT is AutoCloseable, the interface the class implements. CLOSE_OPERATION is close, the required operation marked by @Override. RESOURCE_SEPARATOR is the semicolon between resource declarations inside the try parentheses. It separates map from badge in the initialization order; automatic closure uses the reverse order.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

Common error: Using a comma between the resource declarations. Changing the required close method name. Moving one resource declaration outside the try header.

</details>

### Extend the lifetime model to three resources

In the supplied work cell, add `NamedResource third = new NamedResource("pen")` after the badge declaration inside the same `try` parentheses. Separate resource declarations with semicolons. Preserve the class, existing messages, and normal body.

Before editing and running, list the intended initialization order and predict every output line. Explain which part of the program determines cleanup order.

In [ ]:
Planned initialization order:

Predicted complete three-resource output:

What determines cleanup order:


In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");


Run the complete three-resource program. Record its output and compare it with your prediction. Identify where pen appears in the initialization sequence and where it appears in the cleanup sequence. Use the declaration order to explain the relationship.

In [ ]:
Actual complete three-resource output:

First difference from my prediction, or no difference:

Pen’s initialization and cleanup positions:

Explanation using the resource declarations:


### Test the same resources when the body fails

In that three-resource work cell, add `throw new IllegalStateException("Label work stopped.");` immediately after the `Work` print. Attach a `catch (IllegalStateException problem)` handler after the resource statement. Have it print `Handled: ` followed by `problem.getMessage()`, and keep `Done` after the handler. Preserve the three resource declarations and their order.

Before editing and running, predict the complete report. Place the cleanup messages and handler message in order, and explain whether the body failure changes the cleanup order.

In [ ]:
Predicted complete failure-case output:

Cleanup order and its reason:

Where the handler runs relative to cleanup:


Run the complete failure variation and record its output. Compare the cleanup order with the normal three-resource run. Explain why the handler appears where it does.

Then restore the normal three-resource body by removing the added `throw` statement; the specific handler may remain, but should have no failure to handle. Rerun and record the normal result.

In [ ]:
Actual failure-case output:

Comparison with normal cleanup order:

Why the handler appears at this point:

Actual output after restoring the normal body:


<details>
<summary>Show answer</summary>

The declarations construct map, badge and pen in that order. After Work, automatic cleanup calls close on pen, then badge, then map. Done follows all three. In the failure variation, the body raises IllegalStateException after Work. All three close calls still run in the same reverse order before the attached catch prints Handled: Label work stopped. Done follows the handler. The extra resource extends the sequence without changing the cleanup rule.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge");
     NamedResource third = new NamedResource("pen")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Opened: pen
Work
Closed: pen
Closed: badge
Closed: map
Done
```

Common error: Adding the third object as an ordinary declaration in the body and assuming the header will close it. Closing the first-declared resource first. Placing the handler message before automatic closure.

**Additional test: `Three resources and a body failure`.** A body failure still triggers pen, badge and map cleanup before the specific handler responds. All three constructors had completed successfully.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge");
     NamedResource third = new NamedResource("pen")) {
    System.out.println("Work");
    throw new IllegalStateException("Label work stopped.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Opened: pen
Work
Closed: pen
Closed: badge
Closed: map
Handled: Label work stopped.
Done
```

</details>

### Repair cleanup that a failure skips

Read the supplied faulty program without running it. Its manual `close` calls are ordinary statements at the end of the `try` body, and `stop` is `true`. Predict its complete output and identify which cleanup statements the failure skips.

Plan a repair in the empty Java work cell: copy the whole program, move both resource declarations into a try-with-resources header, and remove the manual close calls. Preserve the class, flag, body, exception, handler, and messages. Predict the repaired output before implementing and running it.

**Incomplete or faulty code for repair — read this example; write your corrected program in the Java work cell.**

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = true;
try {
    NamedResource first = new NamedResource("map");
    NamedResource second = new NamedResource("badge");
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
    second.close();
    first.close();
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

In [ ]:
Predicted faulty output:

Skipped cleanup statements and the statement that skips them:

Planned resource header and removed statements:

Predicted repaired output with stop true:


Run your complete repair with `stop` still `true`. Record the output and compare it with your prediction. Explain why the original matching handler did not ensure cleanup, and identify what your repair changed about the resource lifetime.

In [ ]:
Actual repaired output with stop true:

Difference from my prediction, or no difference:

Why catch alone did not close the resources:

How the repair arranges cleanup:


Change only `stop` to `false` in your repaired program. Before rerunning, predict all output lines and count the expected close calls for each acquired object. Keep the handler and automatic cleanup structure unchanged.

In [ ]:
Predicted normal output:

Expected close-call count for map:
Expected close-call count for badge:

Reason for those counts:


Run the repaired normal case and record its output. Explain why keeping the old manual `close` calls as well as automatic cleanup would add unwanted calls on this path. Then restore `stop` to `true`, rerun the complete program, and record the restored failure case.

In [ ]:
Actual output with stop false:

Why the old manual close calls must be removed:

Actual output after restoring stop true:

Corrections and retest results, if needed:


<details>
<summary>Show answer</summary>

In the faulty version, both constructors and Work run, then the conditional throw leaves the try body. The ordinary second.close and first.close statements are skipped. The catch reports the failure, but that response does not call either missing cleanup operation. The repair registers both initialized objects in a try-with-resources header. Java closes badge then map as control leaves the body, before the catch responds. Removing the old manual close calls prevents extra calls on the successful path. With stop false, there is no body exception; automatic cleanup still closes both once before Done.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = true;
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Handled: Label work stopped.
Done
```

Common error: Keeping the old manual close calls after adding automatic cleanup, causing extra close messages on the normal path. Moving only one declaration into the resource header. Assuming the catch automatically executes skipped ordinary close statements.

**Additional test: `Repaired program with normal work`.** With stop false, the body completes and automatic cleanup runs once per resource in reverse order. Retaining manual closes would add unwanted duplicate Closed lines.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = false;
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

</details>

## Independent Practice

### Build an equipment-return model

Write a complete `LoanResource` class and caller in the next Java work cell. This model prints loan events; it does not control physical equipment. Implement `AutoCloseable`, store a private `String name`, and give the constructor a name parameter. The constructor stores that name and prints `Borrowed: ` plus the name. Override public `void close()` to print `Returned: ` plus the name.

Acquire `"camera"` and then `"tripod"` in one try-with-resources statement. In its body, print `Begin work`, then throw `IllegalStateException` with message `Practice interruption.` Attach a handler for that type which prints `Handled: ` plus the exception message. Print `Finished` once after the handler.

Before writing and running your program, outline the class and caller, and predict the complete output.

In [ ]:
Class responsibilities and method signatures:

Planned resource declarations and caller structure:

Predicted complete output:


Run your complete equipment-return model. Record the output, compare it with your prediction, and identify the beginning and end of each modeled loan. Explain the return order using acquisition order, then explain the handler’s position relative to cleanup.

In [ ]:
Actual complete output:

Difference from my prediction, or no difference:

Camera lifetime endpoints:
Tripod lifetime endpoints:

Reason for return order:

Reason for handler position:


### Test both work paths with one and two resources

Plan four runs of your complete `LoanResource` program. For normal work, remove only the body’s `throw` statement and retain the handler. For a single resource, remove only the tripod declaration and retain camera. Keep the class and other messages unchanged.

Predict the complete output and acquisition/return counts for each case below before running the tests.

In [ ]:
Camera + tripod, failing body — predicted output and counts:

Camera + tripod, normal body — predicted output and counts:

Camera only, failing body — predicted output and counts:

Camera only, normal body — predicted output and counts:


Run all four planned cases, each as a complete program. Record each result and compare it with its prediction. Explain whether a normal body produces a `Handled` line and why `Finished` remains last. Check the number of return messages against the number of acquired objects. Finally, restore the two-resource failure case and record a confirming run.

In [ ]:
Camera + tripod, failing body — actual output and counts:

Camera + tripod, normal body — actual output and counts:

Camera only, failing body — actual output and counts:

Camera only, normal body — actual output and counts:

Why normal work has no Handled line:

Why each acquired object has one return message and Finished is last:

Restored baseline output:


### Keep two cleanup failures alongside the body failure

Use the separate Java work cell for this adaptation. Copy the earlier complete `CleanupResource` class and caller. Change the resource names to `"camera"` and `"tripod"`, set both `failOnClose` flags to `true`, and change the body exception message to `Practice interruption.` Preserve its `Work`, `Closing`, `Primary`, `Suppressed count`, `Suppressed`, and `After cleanup` reports.

Before writing and running the adaptation, predict every output line, including the primary and suppressed reports. Explain the order in which you expect the close operations to be attempted.

In [ ]:
Planned changes to the copied program:

Predicted complete output:

Predicted primary failure:

Predicted suppressed failures in order:

Reason for cleanup-attempt order:


In [ ]:
// Write the complete CleanupResource adaptation here.


Run the complete `CleanupResource` adaptation. Record its full output. Identify where the primary failure starts and where each suppressed failure starts. Explain why both close operations are attempted even though each fails. Compare this case with the earlier empty suppressed array. If the output differs from your prediction, explain the first difference and retain any correction and retest.

In [ ]:
Actual complete output:

Primary failure and its origin:

Suppressed failures in order, with their origins:

Why both close attempts occur:

Comparison with the empty-suppressed-array case:

First difference, corrections, and retest result:


<details>
<summary>Show answer</summary>

LoanResource implements the cleanup interface and prints observable acquisition/return messages. The caller constructs camera and then tripod before Begin work. The body throws Practice interruption. Automatic cleanup calls close on tripod first and camera second. Only after both calls finish does the matching handler print Handled: Practice interruption. Finished follows the handler. This preserves the original body failure while ensuring both successfully initialized classroom resources complete their modeled lifetimes. Removing the throw lets the body finish normally; both return messages still appear but the catch does not run. Keeping only camera produces one acquisition and one return on either work path. The separate CleanupResource adaptation gives both close calls a failure: tripod is attempted first, then camera. The body exception remains primary, while the two cleanup failures are attached in that closing order. The suppressed count is 2. In the earlier comparison with both close flags false, the body still failed but the count was 0, because no cleanup exception occurred.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera");
     LoanResource second = new LoanResource("tripod")) {
    System.out.println("Begin work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Borrowed: tripod
Begin work
Returned: tripod
Returned: camera
Handled: Practice interruption.
Finished
```

Common error: Reversing the required camera-then-tripod construction order. Placing close calls only after the body throw. Printing the handler response before automatic cleanup. Changing the required messages or confusing the printed model with an actual equipment loan.

**Additional test: Two resources with normal work.** Removing the body throw leaves automatic reverse cleanup intact. The catch is skipped and Finished follows both return messages.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera");
     LoanResource second = new LoanResource("tripod")) {
    System.out.println("Begin work");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Borrowed: tripod
Begin work
Returned: tripod
Returned: camera
Finished
```

**Additional test: Single camera resource with a body failure.** Only camera is acquired and automatically returned. Its close finishes before the specific handler prints the body failure.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera")) {
    System.out.println("Begin work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Begin work
Returned: camera
Handled: Practice interruption.
Finished
```

**Additional test: Single camera resource with normal work.** Camera is acquired and returned once. Work succeeds, so there is no Handled line and Finished follows cleanup.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera")) {
    System.out.println("Begin work");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Begin work
Returned: camera
Finished
```

**Additional test: Separate failing-close adaptation: camera and tripod both fail during cleanup.** The body failure Practice interruption. remains primary. Automatic closing attempts tripod and then camera, retaining both failures as suppressed in that order. The count of 2 and two distinct messages prove both close attempts were observed.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("camera", true);
     CleanupResource second = new CleanupResource("tripod", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: camera
Opened: tripod
Work
Closing: tripod
Closing: camera
Primary: Practice interruption.
Suppressed count: 2
Suppressed: Close failed: tripod
Suppressed: Close failed: camera
After cleanup
```

</details>

## Summary

Cleanup belongs to leaving an operation, including its failed paths. A normally completing `finally` preserves a pending return or exception. A try-with-resources header arranges closing for successfully initialized objects in reverse order. If both work and closing fail, the body failure remains primary and the cleanup failures are retained as suppressed exceptions.

### Recall the cleanup rules

Close the answer panels. Explain how cleanup fits into normal completion, a pending return, and a body exception. State which mechanism registers an `AutoCloseable` object for automatic cleanup and how multiple successfully initialized resources are ordered for closure. Finish by distinguishing an earlier exception cause from a suppressed cleanup failure.

In [ ]:
Normal completion:

Pending return:

Body exception:

Registration and closure order:

Cause versus suppressed cleanup failure:


<details><summary>Show answer: connect cleanup to the exit path</summary>

During normal completion, cleanup runs before following statements. During a return, Java evaluates the return value first, runs `finally`, then delivers that value if cleanup completes normally. During a body failure, cleanup runs before the failure continues to a handler. A new return or throw inside `finally` can replace the pending outcome.

Implementing `AutoCloseable` supplies a close operation; declaring the object in a try-with-resources header registers automatic closing. Successfully initialized resources close in reverse initialization order. A cause links an exception to an earlier underlying failure. A suppressed exception records an additional failure, such as a close failure retained alongside a primary body exception. Neither relationship means the diagnostic was discarded.

</details>

## Reflection

A task successfully opens an input stream and then an output stream before writing data. Treat these as two resources registered in that order; you do not need to know their Java APIs yet. Writing then fails.

Describe the cleanup order you would expect and what should happen if closing also fails. Explain which failure information the caller needs to investigate both the original work and cleanup.

In [ ]:
Expected cleanup order and reason:

What should happen if a close operation fails too:

Failure information the caller needs and why:


## Looking Ahead

You have traced successful operations, failures, and cleanup by reading output. Next, automated behavior tests will compare actual results with expected results and check that invalid inputs raise the intended exceptions.

## Supplemental Reading

- [Catching exceptions and cleanup](https://dev.java/learn/exceptions/catching-handling/) connects handlers, finally, and automatic resource management.
- [Java 21 try-with-resources rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-14.html#jls-14.20.3) specifies initialization, closure, and suppressed failures.
- [Java 21 AutoCloseable API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/AutoCloseable.html) defines the cleanup interface.
- [Java 21 suppressed exceptions](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Throwable.html#getSuppressed()) documents how to inspect secondary failures.
